# Matrix-weighted localization in $SE(3)$

This notebook estimates three-dimensional poses from perturbed observations of known landmarks and relative-pose measurements. Each landmark observation has a full anisotropic covariance.

The committed dataset was generated by [MatrixWeightedLocalizationExample.py](MatrixWeightedLocalizationExample.py). It stores conventional ground-truth poses $wTk$, while the matrix-weighted graph optimizes $kTw=(wTk)^{-1}$.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415  
All Rights Reserved  
Authors: Frank Dellaert, et al. (see THANKS for the full author list)  
See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/MatrixWeightedLocalizationExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import numpy as np
import gtsam
import plotly.graph_objects as go

## Measurement and frame conversions

The g2o dataset stores $wTk$, known $wL$, relative-pose factors with $iTj=(wTi)^{-1}wTj$, and Cartesian landmark observations encoded as bearing-range factors.

For unit bearing $u$, range $r$, and measured Unit3 tangent basis $B$,

$$kP=ru, \qquad J=\begin{bmatrix}rB & u\end{bmatrix}, \qquad \Sigma_{kP}=J\Sigma_{br}J^{\mathsf T}.$$

This covariance conversion is exact at the measurement linearization point; the nonlinear Cartesian and bearing-range Gaussian models are not globally identical.

The matrix-weighted known-landmark residual is $kRw\,wL+k\mathbf{t}_w-\widetilde{kP}$. For $iTw=(wTi)^{-1}$ and $jTw=(wTj)^{-1}$, the loaded measurement satisfies $iTw=iTj\,jTw$, giving the left residual $\operatorname{vec}(iTw-iTj\,jTw)$.

In [3]:
data_file = gtsam.findExampleDataFile(
    "matrix_weighted_localization.g2o"
)
source_graph, dataset_values = gtsam.readG2o(data_file, is3D=True)
print(f"loaded factors: {source_graph.size()}")
print(f"loaded values:  {dataset_values.size()}")

loaded factors: 14
loaded values:  7


## Build the matrix-weighted graph

The loaded graph supplies all connectivity. Bearing-range factors become unary known-landmark factors after recovering $kP$ and its Cartesian covariance. Pose3 between factors become left Frobenius factors with the same keys, $iTj$, and noise model. Loaded $wTk$ values are inverted before insertion as matrix-weighted ground truth.

In [4]:
graph = gtsam.NonlinearFactorGraph()
pose_keys, landmark_keys = set(), set()
cartesian_covariances = []

for index in range(source_graph.size()):
    factor = source_graph.at(index)
    keys = tuple(factor.keys())
    if isinstance(factor, gtsam.BearingRangeFactor3D):
        pose_key, landmark_key = keys
        measured = factor.measured()
        range_, bearing = measured.range(), measured.bearing()
        kP = range_ * bearing.unitVector()
        J = np.column_stack(
            (range_ * bearing.basis(), bearing.unitVector())
        )
        covariance = J @ factor.noiseModel().covariance() @ J.T
        graph.add(
            gtsam.KnownLandmarkFactorPose3(
                pose_key,
                dataset_values.atPoint3(landmark_key),
                kP,
                gtsam.noiseModel.Gaussian.Covariance(covariance),
            )
        )
        pose_keys.add(pose_key)
        landmark_keys.add(landmark_key)
        cartesian_covariances.append(covariance)
    elif isinstance(factor, gtsam.BetweenFactorPose3):
        i, j = keys
        graph.add(
            gtsam.FrobeniusLeftBetweenFactorPose3(
                i, j, factor.measured(), factor.noiseModel()
            )
        )
        pose_keys.update(keys)
    else:
        raise TypeError(f"Unexpected factor type: {type(factor).__name__}")

pose_keys, landmark_keys = sorted(pose_keys), sorted(landmark_keys)
ground_truth = gtsam.Values()
for key in pose_keys:
    ground_truth.insert(key, dataset_values.atPose3(key).inverse())

eigenvalues = np.linalg.eigvalsh(cartesian_covariances[0])
anisotropicity = np.sqrt(eigenvalues[-1] / eigenvalues[0])
print(f"matrix-weighted factors: {graph.size()}")
print("covariance eigenvalues:", eigenvalues)
print(f"sqrt condition number: {anisotropicity:.1f}")
assert np.isclose(anisotropicity, 10.0)

matrix-weighted factors: 14
covariance eigenvalues: [1.e-04 1.e-04 1.e-02]
sqrt condition number: 10.0


## Perturb and optimize

Initialization also uses $kTw=(wTk)^{-1}$. Because the measurements are perturbed, ground truth has nonzero objective and the maximum-likelihood estimate need not equal it. We require optimization to improve both the initial and ground-truth objectives while remaining close to the stored trajectory.

In [5]:
perturbation = np.array([0.02, -0.015, 0.01, 0.08, -0.05, 0.06])
initial = gtsam.Values()
for index, key in enumerate(pose_keys):
    kTw = dataset_values.atPose3(key).inverse()
    initial.insert(key, kTw.retract((index + 1) * perturbation))

parameters = gtsam.GaussNewtonParams()
parameters.setMaxIterations(100)
parameters.setRelativeErrorTol(1e-12)
result = gtsam.GaussNewtonOptimizer(
    graph, initial, parameters
).optimize()

ground_truth_error = graph.error(ground_truth)
initial_error = graph.error(initial)
final_error = graph.error(result)
pose_errors = np.array([
    np.linalg.norm(
        dataset_values.atPose3(key).localCoordinates(
            result.atPose3(key).inverse()
        )
    )
    for key in pose_keys
])

print(f"ground-truth error: {ground_truth_error:.3e}")
print(f"initial error:      {initial_error:.3e}")
print(f"final error:        {final_error:.3e}")
print(f"mean pose error:    {pose_errors.mean():.3e}")
print(f"maximum pose error: {pose_errors.max():.3e}")

assert final_error < initial_error
assert final_error < ground_truth_error
assert pose_errors.max() < 0.05

ground-truth error: 2.415e+01
initial error:      6.873e+03
final error:        1.779e+01
mean pose error:    1.614e-02
maximum pose error: 1.992e-02


In [6]:
# Convert kTw estimates back to conventional wTk for comparison.
ground_truth_wPs = np.vstack([
    dataset_values.atPose3(key).translation() for key in pose_keys
])
initial_wPs = np.vstack([
    initial.atPose3(key).inverse().translation() for key in pose_keys
])
result_wPs = np.vstack([
    result.atPose3(key).inverse().translation() for key in pose_keys
])
wLs = np.vstack([
    dataset_values.atPoint3(key) for key in landmark_keys
])

fig = go.Figure()
fig.add_scatter(
    x=initial_wPs[:, 0], y=initial_wPs[:, 1], mode="lines+markers",
    line={"dash": "dash"}, name="initial poses",
)
fig.add_scatter(
    x=ground_truth_wPs[:, 0], y=ground_truth_wPs[:, 1],
    mode="lines+markers", line={"color": "black"},
    name="ground truth",
)
fig.add_scatter(
    x=result_wPs[:, 0], y=result_wPs[:, 1], mode="lines+markers",
    marker={"symbol": "x"}, name="optimized poses",
)
fig.add_scatter(
    x=wLs[:, 0], y=wLs[:, 1], mode="markers",
    marker={"symbol": "star", "size": 12},
    name="known landmarks",
)
fig.update_layout(
    title="Matrix-weighted localization",
    xaxis_title="world x [m]", yaxis_title="world y [m]",
    template="plotly_white", width=700, height=500,
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

## Reading the result

The recovered covariance eigenvalues are $\sigma_l^2,\sigma_l^2,\sigma_r^2$, so their square-root condition number recovers the anisotropicity of 10. The full covariance is generally not diagonal because its radial eigenvector follows the observation ray.

The optimizer returns $kTw$, whereas the dataset stores $wTk$. We invert every estimate before comparison. For this deterministic noisy dataset, the optimized likelihood is better than the likelihood at ground truth, while the poses remain within a few centimeters of the generating trajectory.